In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, ConfusionMatrixDisplay)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from imblearn.over_sampling import SMOTE
import warnings; warnings.filterwarnings('ignore')

# Load dữ liệu
df = pd.read_csv('creditcard.csv')
print('Shape:', df.shape)
print('\nPhân bố nhãn:')
print(df['Class'].value_counts())
print('Tỷ lệ gian lận: {:.4f}%'.format(
    df['Class'].sum() / len(df) * 100))


In [1]:
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE

# Load data
df = pd.read_csv('creditcard.csv')

# Scale Amount và Time TRƯỚC
scaler_amount = StandardScaler()
scaler_time   = StandardScaler()
df['scaled_amount'] = scaler_amount.fit_transform(df[['Amount']])
df['scaled_time']   = scaler_time.fit_transform(df[['Time']])
df.drop(['Amount', 'Time'], axis=1, inplace=True)

X = df.drop('Class', axis=1)
y = df['Class']

print("Feature columns:", X.columns.tolist())
# Phải ra: ['V1'...'V28', 'scaled_amount', 'scaled_time']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

# Train lại
rf_new = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_new.fit(X_train_sm, y_train_sm)

# Kiểm tra ngay
from sklearn.metrics import classification_report
y_pred = rf_new.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['Bình thường', 'Gian lận']))

# Test dòng gian lận thủ công
X_fraud_test = X_test[y_test == 1].head(3)
proba = rf_new.predict_proba(X_fraud_test)
print("Xác suất gian lận (phải cao):", proba[:, 1])

Feature columns: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'scaled_amount', 'scaled_time']
              precision    recall  f1-score   support

 Bình thường       1.00      1.00      1.00     56864
    Gian lận       0.82      0.82      0.82        98

    accuracy                           1.00     56962
   macro avg       0.91      0.91      0.91     56962
weighted avg       1.00      1.00      1.00     56962

Xác suất gian lận (phải cao): [0.94 1.   0.84]


In [2]:
# Export bundle đúng
bundle = {
    "model":         rf_new,
    "scaler_amount": scaler_amount,
    "scaler_time":   scaler_time,
}
joblib.dump(bundle, r"D:\CDTN\fraud_api\ml_models\rf_bundle_v2.pkl")
print("Đã lưu rf_bundle_v2.pkl!")

Đã lưu rf_bundle_v2.pkl!


In [ ]:
# Thống kê mô tả
print(df.describe())

# Kiểm tra missing values
print('Missing values:', df.isnull().sum().sum())

# Trực quan hóa phân bố lớp
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Biểu đồ cột
counts = df['Class'].value_counts()
axes[0].bar(['Bình thường (0)', 'Gian lận (1)'],
            counts.values, color=['steelblue', 'tomato'])
axes[0].set_title('Phân bố giao dịch')
axes[0].set_ylabel('Số lượng')

# Phân bố Amount theo lớp
df[df['Class'] == 0]['Amount'].hist(
    ax=axes[1], bins=50, alpha=0.6, label='Bình thường', color='steelblue')
df[df['Class'] == 1]['Amount'].hist(
    ax=axes[1], bins=50, alpha=0.8, label='Gian lận', color='tomato')
axes[1].set_title('Phân bố số tiền giao dịch')
axes[1].legend()
plt.tight_layout(); plt.show()


In [ ]:
# Chuẩn hóa Amount và Time
scaler = StandardScaler()
df['scaled_amount'] = scaler.fit_transform(df[['Amount']])
df['scaled_time']   = scaler.fit_transform(df[['Time']])
df.drop(['Amount', 'Time'], axis=1, inplace=True)

# Tách features và target
X = df.drop('Class', axis=1)
y = df['Class']

# Tách train / test (80 / 20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print('Train size:', X_train.shape)
print('Test size: ', X_test.shape)

# Áp dụng SMOTE trên tập train
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print('\nSau SMOTE – phân bố y_train:')
print(pd.Series(y_train_sm).value_counts())


In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sm, y_train_sm)
y_pred_lr = lr.predict(X_test)

print('=== LOGISTIC REGRESSION ===')
print(confusion_matrix(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr,
      target_names=['Bình thường', 'Gian lận']))


In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_sm, y_train_sm)
y_pred_rf = rf.predict(X_test)

print('=== RANDOM FOREST ===')
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf,
      target_names=['Bình thường', 'Gian lận']))


In [ ]:
import os
import joblib

# 1. Khai báo đường dẫn tuyệt đối đến thư mục đích
duong_dan_dich = r"D:\CDTN\fraud_api\ml_models\random_forest_fraud.pkl"

# 2. Kiểm tra và tự động tạo thư mục nếu chưa tồn tại
thu_muc_dich = os.path.dirname(duong_dan_dich)
if not os.path.exists(thu_muc_dich):
    os.makedirs(thu_muc_dich)

# 3. Lưu mô hình 'rf' vào đường dẫn trên
joblib.dump(rf, duong_dan_dich)

print(f" Đã lưu mô hình Random Forest thành công vào: {duong_dan_dich}")

In [ ]:
import joblib, pandas as pd
from sklearn.preprocessing import StandardScaler

df_raw = pd.read_csv('creditcard.csv')

scaler_amount = StandardScaler()
scaler_time   = StandardScaler()
scaler_amount.fit(df_raw[['Amount']])
scaler_time.fit(df_raw[['Time']])

bundle = {
    "model":         rf,             # model đã train
    "scaler_amount": scaler_amount,
    "scaler_time":   scaler_time,
}
joblib.dump(bundle, r"D:\CDTN\fraud_api\ml_models\rf_bundle.pkl")
print("Done!")

In [ ]:
iso = IsolationForest(contamination=0.0017, random_state=42)
iso.fit(X_train)
y_raw = iso.predict(X_test)
y_pred_iso = [1 if v == -1 else 0 for v in y_raw]

print('=== ISOLATION FOREST ===')
print(confusion_matrix(y_test, y_pred_iso))
print(classification_report(y_test, y_pred_iso,
      target_names=['Bình thường', 'Gian lận']))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
models   = ['Logistic Regression', 'Random Forest', 'Isolation Forest']
preds    = [y_pred_lr, y_pred_rf, y_pred_iso]
colors   = ['Blues', 'Greens', 'Oranges']

for ax, name, pred, cmap in zip(axes, models, preds, colors):
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(cm,
               display_labels=['Bình thường', 'Gian lận'])
    disp.plot(ax=ax, colorbar=False, cmap=cmap)
    ax.set_title(name)

plt.suptitle('So sánh Confusion Matrix – 3 Mô hình', fontsize=14)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
param_grid = {
    'n_estimators':     [100, 200, 300],
    'max_depth':        [10, 20, None],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train_sm, y_train_sm)

print('Best params:', grid_search.best_params_)
print('Best F1    :', grid_search.best_score_)

# Đánh giá mô hình tối ưu
best_rf = grid_search.best_estimator_
y_pred_best = best_rf.predict(X_test)
print(classification_report(y_test, y_pred_best,
      target_names=['Bình thường', 'Gian lận']))


In [ ]:
importance = rf.feature_importances_
feat_df = pd.DataFrame({
    'feature':    X.columns,
    'importance': importance
}).sort_values('importance', ascending=False)

# In top 10
print(feat_df.head(10).to_string(index=False))

# Biểu đồ
plt.figure(figsize=(10, 6))
sns.barplot(data=feat_df.head(10), x='importance', y='feature',
            palette='viridis')
plt.title('Top 10 Feature Importance – Random Forest')
plt.xlabel('Mức độ quan trọng')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()


In [1]:
import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

bundle = joblib.load(r"D:\CDTN\fraud_api\ml_models\rf_bundle_v2.pkl")
model = bundle['model']
sa    = bundle['scaler_amount']
st    = bundle['scaler_time']

# Load data gốc
df = pd.read_csv('creditcard.csv')
df['scaled_amount'] = sa.transform(df[['Amount']])
df['scaled_time']   = st.transform(df[['Time']])
df.drop(['Amount','Time'], axis=1, inplace=True)

X = df.drop('Class', axis=1)
y = df['Class']

_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Lấy 5 giao dịch gian lận thật từ test set
fraud_samples = X_test[y_test == 1].head(5)
proba = model.predict_proba(fraud_samples)[:,1]
print("Xác suất 5 giao dịch gian lận thật:", proba)

# In ra 1 mẫu gian lận để test API
sample = fraud_samples.iloc[0]
print("\nMẫu gian lận để test API:")
for col in sample.index:
    print(f'  "{col}": {sample[col]:.6f}')

Xác suất 5 giao dịch gian lận thật: [0.94 1.   0.84 0.82 0.93]

Mẫu gian lận để test API:
  "V1": -1.271244
  "V2": 2.462675
  "V3": -2.851395
  "V4": 2.324480
  "V5": -1.372245
  "V6": -0.948196
  "V7": -3.065234
  "V8": 1.166927
  "V9": -2.268771
  "V10": -4.881143
  "V11": 2.255147
  "V12": -4.686387
  "V13": 0.652375
  "V14": -6.174288
  "V15": 0.594380
  "V16": -4.849692
  "V17": -6.536521
  "V18": -3.119094
  "V19": 1.715494
  "V20": 0.560478
  "V21": 0.652941
  "V22": 0.081931
  "V23": -0.221348
  "V24": -0.523582
  "V25": 0.224228
  "V26": 0.756335
  "V27": 0.632800
  "V28": 0.250187
  "scaled_amount": -0.353189
  "scaled_time": -0.796134
